# 20 — Tuning de CatBoost con Optuna (TPE)

**TPE** (*Tree-structured Parzen Estimator*, no está en el material): a diferencia de la grilla o del azar, aprende de las pruebas anteriores.
1. Arranca con ~10 combinaciones al azar.
2. Separa las pruebas en "buenas" (las de mejor accuracy) y "malas".
3. Para cada hiperparámetro estima la distribución de valores en las buenas, l(x), y en las malas, g(x).
4. Propone la combinación que maximiza l(x)/g(x): valores frecuentes en las buenas y raros en las malas.
5. La prueba y repite.

**Optuna** es la librería que lo implementa.

**Modelo:** CatBoost del modelo final (`armar_pipeline_final()`, con `duracion_dias`; 0,9414 en CV).

**Espacio de búsqueda:**
- `depth` 4–8;
- `learning_rate` 0,02–0,2 (log);
- `l2_leaf_reg` 1–30 (log);
- `random_strength` 0,1–10 (log);
- `iterations` lo decide el *early stopping* (paciencia 100, tope 3.000) sobre el 10% del train de cada fold, **nunca** del fold de validación.

La primera prueba es la configuración por defecto (como referencia dentro del estudio). 35 pruebas, CV de siempre, optimizando accuracy. Los resultados se guardan en `.cache_laboratorio/optuna_catboost.db` (sqlite), así un corte no pierde lo hecho.

**Confirmación del ganador** (al final): para no perder el 10% que usa el *early stopping* (costo que vimos en C2), se reentrena con el **100% del train del fold** y la cantidad de árboles que encontró el *early stopping*, sin validación interna. Después: comparación pareada contra CatBoost con duración, con las semillas de CV 42 y 7.

In [ ]:
import sys
sys.path.append("..")
import os, time, warnings
import numpy as np
import pandas as pd
import optuna
from joblib import Memory
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from catboost import CatBoostClassifier

from features import (aplanar_anidadas, agregar_features_contenido, agregar_categoricas_crudas,
                      armar_pipeline_final, PrediccionOOF, armar_modelo_texto, INDICES_CAT)

SEED = 42
crudo = pd.read_json("../data/train_data.jsonlines", lines=True)
y = (crudo["condition"] == "new").astype(int)
train = agregar_categoricas_crudas(agregar_features_contenido(aplanar_anidadas(crudo)))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
carpeta_lab = ".cache_laboratorio"
memoria_texto = Memory(location=carpeta_lab, verbose=0)


class CatBoostConEarlyStopping(ClassifierMixin, BaseEstimator):
    """CatBoost con early stopping sobre el 10% del train que recibe (nunca del fold de validación)."""

    def __init__(self, depth=6, learning_rate=None, l2_leaf_reg=3, random_strength=1, iterations=3000,
                 early_stopping_rounds=100, validation_fraction=0.1, random_state=42):
        self.depth = depth
        self.learning_rate = learning_rate
        self.l2_leaf_reg = l2_leaf_reg
        self.random_strength = random_strength
        self.iterations = iterations
        self.early_stopping_rounds = early_stopping_rounds
        self.validation_fraction = validation_fraction
        self.random_state = random_state

    def fit(self, X, y, cat_features=None):
        X_e, X_v, y_e, y_v = train_test_split(X, y, test_size=self.validation_fraction, stratify=y,
                                              random_state=self.random_state)
        self.modelo_ = CatBoostClassifier(depth=self.depth, learning_rate=self.learning_rate, l2_leaf_reg=self.l2_leaf_reg,
                                          random_strength=self.random_strength, iterations=self.iterations,
                                          random_state=self.random_state, verbose=0)
        self.modelo_.fit(X_e, y_e, cat_features=cat_features, eval_set=(X_v, y_v),
                         early_stopping_rounds=self.early_stopping_rounds, use_best_model=True)
        self.classes_ = self.modelo_.classes_
        self.n_iter_ = self.modelo_.get_best_iteration() + 1
        return self

    def predict_proba(self, X):
        return self.modelo_.predict_proba(X)

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]


def con_texto_cacheado(pipeline):
    transformers = pipeline.named_steps["preprocesador"].transformers
    i = [n for n, _, _ in transformers].index("texto")
    transformers[i] = ("texto", Pipeline([("oof", PrediccionOOF(armar_modelo_texto(), random_state=SEED)),
                                          ("identidad", "passthrough")], memory=memoria_texto), ["title", "warranty_texto"])
    return pipeline


def pipeline_con(modelo):
    pipeline = con_texto_cacheado(armar_pipeline_final())
    pipeline.steps[-1] = ("modelo", modelo)
    return pipeline

## El estudio (35 pruebas)

In [ ]:
def objetivo(trial):
    modelo = CatBoostConEarlyStopping(
        depth=trial.suggest_int("depth", 4, 8),
        learning_rate=trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
        l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1, 30, log=True),
        random_strength=trial.suggest_float("random_strength", 0.1, 10, log=True),
        random_state=SEED,
    )
    inicio = time.time()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        r = cross_validate(pipeline_con(modelo), train, y, cv=cv, scoring="accuracy", n_jobs=5,
                           params={"modelo__cat_features": INDICES_CAT}, return_estimator=True)
    trial.set_user_attr("accuracy_folds", r["test_score"].tolist())
    trial.set_user_attr("arboles_folds", [int(e.named_steps["modelo"].n_iter_) for e in r["estimator"]])
    trial.set_user_attr("minutos", (time.time() - inicio) / 60)
    return r["test_score"].mean()


estudio = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED),
                              study_name="catboost_p6b_duracion", storage=f"sqlite:///{carpeta_lab}/optuna_catboost.db",
                              load_if_exists=True)
N_PRUEBAS = 35
if len(estudio.trials) == 0:
    estudio.enqueue_trial({"depth": 6, "learning_rate": 0.0575, "l2_leaf_reg": 3, "random_strength": 1})
optuna.logging.set_verbosity(optuna.logging.WARNING)
inicio = time.time()
estudio.optimize(objetivo, n_trials=max(0, N_PRUEBAS - len([t for t in estudio.trials if t.state.is_finished()])))
print(f"Estudio: {(time.time() - inicio) / 60:.1f} min | pruebas terminadas: {len(estudio.trials)}")

In [ ]:
tabla = estudio.trials_dataframe(attrs=("number", "value", "params", "user_attrs"))
tabla = tabla.rename(columns=lambda c: c.replace("params_", "").replace("user_attrs_", ""))
tabla["arboles_mediana"] = tabla["arboles_folds"].map(np.median)
columnas = ["number", "value", "depth", "learning_rate", "l2_leaf_reg", "random_strength", "arboles_mediana", "minutos"]
print("Mejor prueba:", estudio.best_trial.number, estudio.best_params, "| accuracy", round(estudio.best_value, 4))
tabla[columnas].sort_values("value", ascending=False).head(12).round(4)

## Resultado y conclusión (el estudio se cerró en 24 pruebas)

**Por qué 24 y no 35.** La máquina (18 GB de RAM) se saturó: 5 CatBoost en paralelo, cada uno intentando usar todos los núcleos, más otra notebook corriendo al mismo tiempo. Los procesos de trabajo se caían y se volvían a crear, y una sola prueba pasó a tardar horas. Las 24 pruebas terminadas quedaron guardadas en `.cache_laboratorio/optuna_catboost.db`. Las celdas de arriba quedaron sin output por ese corte.

Las 10 mejores (accuracy con *early stopping* sobre el 10% del train de cada fold; la prueba 0 es la configuración por defecto):

| prueba | accuracy (con early stopping) | depth | learning_rate | l2_leaf_reg | random_strength | árboles (mediana) | minutos |
|---|---|---|---|---|---|---|---|
| 18 | 0.9411 | 7 | 0.021 | 2.97 | 0.11 | 2936 | 5.7 |
| 17 | 0.9408 | 7 | 0.021 | 1.10 | 1.54 | 2321 | 9.9 |
| 6 | 0.9407 | 7 | 0.028 | 2.70 | 0.54 | 2203 | 4.8 |
| 0 | 0.9406 | 6 | 0.058 | 3.00 | 1.00 | 1235 | 2.8 |
| 4 | 0.9406 | 8 | 0.033 | 1.86 | 0.23 | 1575 | 5.5 |
| 11 | 0.9406 | 6 | 0.020 | 1.93 | 0.73 | 2863 | 4.2 |
| 7 | 0.9405 | 6 | 0.122 | 1.97 | 1.07 | 705 | 1.4 |
| 13 | 0.9405 | 6 | 0.024 | 4.63 | 1.45 | 2746 | 4.2 |
| 16 | 0.9405 | 7 | 0.054 | 3.33 | 0.52 | 1151 | 2.4 |
| 20 | 0.9405 | 7 | 0.032 | 1.25 | 1.24 | 1242 | 3.3 |

- **Mejor:** prueba 18 (`depth` 7, `learning_rate` 0,021, `l2_leaf_reg` 3,0, `random_strength` 0,11) = 0,9411, contra **0,9406 de la configuración por defecto en las mismas condiciones**: +0,0005. Es del tamaño del ruido y del sesgo de selección (la mejor de 24 sobre los mismos folds).
- TPE se concentró en `learning_rate` ≈ 0,02 con `depth` 7. Necesita ~2.900 árboles (casi el tope de 3.000) contra ~1.200 del default.
- **Confirmación no realizada:** reentrenar al ganador con el 100% del train del fold (3.300 árboles fijos) tarda más de 30 minutos **por fold** en esta máquina (más de 5 horas para las semillas 42 y 7). No se justifica para +0,0005.
- **Conclusión: se mantiene CatBoost con los parámetros por defecto** (0,9414 con reentrenamiento completo), igual que en C2 y como dice la slide 68 de `Árboles.pdf`. TPE no está en el material.

## Confirmación del ganador (con la máquina libre)

Después de liberar memoria, un fold del ganador con reentrenamiento completo tarda 2,2 min (fold 1: 0,9423 contra 0,9415 del default), así que la confirmación se pudo hacer.

- **CatBoost tuneado:** prueba 18, reentrenado con el **100% del train de cada fold**, con 3.300 árboles fijos (mediana de árboles del *early stopping*, 2.936, dividida por 0,9), sin validación interna.
- **Referencia:** CatBoost por defecto (+ duración), el del modelo final.

Las dos semillas de CV (42 y 7), folds de a uno con todos los núcleos. Si el tuneado pasa, se prueba dentro de P7c (promedio 50/50 con el HistGB P7a + duración).

In [2]:
from sklearn.model_selection import cross_val_predict
from features import armar_pipeline_histgb_combos, agregar_combos

PARAMS_GANADOR = {"depth": 7, "learning_rate": 0.02051048890596459, "l2_leaf_reg": 2.9744291550817916,
                  "random_strength": 0.11371366218880984}
ARBOLES = 3300
train_c = agregar_combos(train)


def histgb_cacheado():
    return con_texto_cacheado(armar_pipeline_histgb_combos())


def oof(pipeline, cv_s, params=None):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return cross_val_predict(pipeline, train_c, y, cv=cv_s, method="predict_proba", n_jobs=1, params=params)[:, 1]


resultados = {}
for semilla in [42, 7]:
    cv_s = StratifiedKFold(n_splits=5, shuffle=True, random_state=semilla)
    fold = np.empty(len(train_c), dtype=int)
    for k, (_, v) in enumerate(cv_s.split(train_c, y)):
        fold[v] = k + 1
    inicio = time.time()
    if semilla == 42:
        anterior = pd.read_csv(os.path.join(carpeta_lab, "oof_p7c_duracion.csv"))
        prob_default, prob_hgb = anterior["catboost"].to_numpy(), anterior["histgb"].to_numpy()
    else:
        prob_default = oof(pipeline_con(CatBoostClassifier(random_state=SEED, verbose=0)), cv_s,
                           {"modelo__cat_features": INDICES_CAT})
        prob_hgb = oof(histgb_cacheado(), cv_s)
    prob_tuneado = oof(pipeline_con(CatBoostClassifier(**PARAMS_GANADOR, iterations=ARBOLES, random_state=SEED, verbose=0)),
                       cv_s, {"modelo__cat_features": INDICES_CAT})
    pd.DataFrame({"fold": fold, "y": y.to_numpy(), "catboost_default": prob_default, "catboost_tuneado": prob_tuneado,
                  "histgb": prob_hgb}).to_csv(os.path.join(carpeta_lab, f"oof_tuning_semilla{semilla}.csv"), index=False)

    def acc(prob):
        return pd.Series((prob >= 0.5).astype(int) == y.to_numpy()).groupby(fold).mean().to_numpy()

    a_def, a_tun = acc(prob_default), acc(prob_tuneado)
    a_p7c, a_p7c_tun = acc(0.5 * prob_default + 0.5 * prob_hgb), acc(0.5 * prob_tuneado + 0.5 * prob_hgb)
    for nombre, a, b in [("CatBoost tuneado vs default", a_tun, a_def), ("P7c con tuneado vs P7c", a_p7c_tun, a_p7c)]:
        d = np.round(b * 0 + a - b, 10)
        gana = int((d > 0).sum())
        resultados[(semilla, nombre)] = {"B": a.mean(), "A": b.mean(), "dif_media": d.mean(), "dif_desvio": d.std(),
                                         "folds_gana": gana, "pasa": gana == 5 or (gana == 4 and d.mean() > d.std())}
        print(f"semilla {semilla} | {nombre}: {a.mean():.4f} vs {b.mean():.4f} | dif por fold {d} | gana {gana}/5")
    print(f"semilla {semilla}: {(time.time() - inicio) / 60:.1f} min")
pd.DataFrame(resultados).T.round(4)

semilla 42 | CatBoost tuneado vs default: 0.9417 vs 0.9414 | dif por fold [ 0.00142857 -0.00085714  0.00021429  0.00014286  0.00078571] | gana 4/5
semilla 42 | P7c con tuneado vs P7c: 0.9420 vs 0.9420 | dif por fold [ 7.142860e-05  1.428571e-04 -2.142857e-04 -7.857143e-04  6.428571e-04] | gana 3/5
semilla 42: 420.0 min


semilla 7 | CatBoost tuneado vs default: 0.9408 vs 0.9406 | dif por fold [ 0.          0.0005      0.001       0.00028571 -0.00078571] | gana 3/5
semilla 7 | P7c con tuneado vs P7c: 0.9423 vs 0.9422 | dif por fold [ 0.          0.00021429  0.00021429  0.00028571 -0.00021429] | gana 3/5
semilla 7: 1745.2 min


B         A dif_media dif_desvio  \
42 CatBoost tuneado vs default  0.941743    0.9414  0.000343   0.000758   
   P7c con tuneado vs P7c       0.941971     0.942 -0.000029   0.000469   
7  CatBoost tuneado vs default    0.9408    0.9406    0.0002   0.000591   
   P7c con tuneado vs P7c       0.942257  0.942157    0.0001   0.000184   

                               folds_gana   pasa  
42 CatBoost tuneado vs default          4  False  
   P7c con tuneado vs P7c               3  False  
7  CatBoost tuneado vs default          3  False  
   P7c con tuneado vs P7c               3  False

**Conclusión de la confirmación: el CatBoost tuneado no pasa el criterio.**

| comparación | semilla 42 | semilla 7 |
|---|---|---|
| CatBoost tuneado vs por defecto | 0,9417 vs 0,9414; 4/5, media +0,0003 < desvío 0,0008 → no pasa | 0,9408 vs 0,9406; 3/5 → no pasa |
| P7c con el tuneado vs P7c actual | 0,9420 vs 0,9420; 3/5, media −0,0000 → no pasa | 0,9423 vs 0,9422; 3/5 → no pasa |

- Con reentrenamiento completo, el tuneado queda a la par del default: +0,0002/0,0003 de media, sin consistencia entre folds. Dentro del promedio con el HistGB la diferencia desaparece.
- Además, usa ~3 veces más árboles y tarda varias veces más en entrenar.
- **Se mantiene CatBoost con los parámetros por defecto en el modelo final** (como en C2 y la slide 68 de `Árboles.pdf`).
- Tiempo real de la confirmación: ~36 horas, porque la máquina estuvo sin memoria libre (otras aplicaciones abiertas). Con la máquina libre, un fold tardaba 2,2 min (~30 min en total). Las probabilidades OOF quedaron en `.cache_laboratorio/oof_tuning_semilla{42,7}.csv`.